# Quantitative Trading: Portfolio Optimization
## Markowitz Mean-Variance + Black-Litterman + Shrinkage Estimation

This notebook implements modern portfolio optimization with focus on handling **estimation error in covariance matrices**—a critical practical problem.

### Nobel Prize Context
- **Harry Markowitz** (Nobel 1990): "The investor should maximize expected return subject to an upper limit on variance."
- **Ledoit & Wolf** (2004): Showed sample covariance is severely ill-conditioned in high dimensions.
- **Black & Litterman** (1991): Incorporated investor views into expected returns using Bayesian framework.

### Key Insight
> **The mean-variance problem is theoretically elegant but practically brittle. Portfolio optimization is dominated by estimation error in $\Sigma$ (covariance) and $\mu$ (returns). Robust methods like shrinkage estimation and Black-Litterman significantly improve real-world performance.**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import inv
import seaborn as sns

sns.set_style('whitegrid')
np.random.seed(42)

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

## Part 1: Markowitz Mean-Variance Framework

### 1.1 Problem Formulation

**Minimize** portfolio variance:
$$\min_w \mathbf{w}^T \Sigma \mathbf{w}$$

**Subject to**:
- Expected return constraint: $\mathbf{w}^T \boldsymbol{\mu} = \mu_p$ (target return)
- Budget constraint: $\sum_i w_i = 1$
- Box constraints: $w_i^{\min} \leq w_i \leq w_i^{\max}$ (long-only: $w_i \geq 0$)

**Solution**: By Lagrangian, the efficient frontier is:
$$\mathbf{w}^* = \frac{1}{\Sigma^{-1}(\mu_p - r_f)} \text{ (with Lagrange multipliers)}$$


In [ ]:
class PortfolioOptimizer:
    """
    Portfolio optimization with multiple approaches:
    - Standard Markowitz (sample covariance)
    - Ledoit-Wolf shrinkage
    - Black-Litterman with investor views
    """
    
    def __init__(self, returns: np.ndarray):
        """
        Parameters:
        -----------
        returns : (T, n) array
            T historical returns for n assets
        """
        self.returns = returns
        self.T, self.n = returns.shape
        
        # Sample statistics
        self.mu_sample = np.mean(returns, axis=0)
        self.sigma_sample = np.cov(returns.T)
        self.rf = 0.02  # Risk-free rate
        
    def ledoit_wolf_shrinkage(self) -> np.ndarray:
        """
        Ledoit-Wolf shrinkage: blend sample covariance with structured target.
        
        Σ_shrunk = (1 - α) Σ_sample + α Σ_target
        
        Where Σ_target is a diagonal matrix (identity scaled).
        Optimal α minimizes expected loss (Frobenius norm).
        """
        # Use identity matrix as target
        target = np.eye(self.n) * np.mean(np.diag(self.sigma_sample))
        
        # Estimate optimal shrinkage intensity
        # Simplified version (exact formula in original paper)
        trace_ss = np.trace(self.sigma_sample @ self.sigma_sample) / self.n
        trace_s = np.trace(self.sigma_sample) / self.n
        
        numerator = trace_s - trace_ss
        denominator = (1 + 1/self.n) * trace_s - trace_ss
        alpha = max(0, min(1, numerator / max(denominator, 1e-8)))
        
        sigma_shrunk = (1 - alpha) * self.sigma_sample + alpha * target
        
        return sigma_shrunk, alpha
    
    def black_litterman(self, views: np.ndarray, view_confidence: np.ndarray,
                       market_caps: np.ndarray = None) -> np.ndarray:
        """
        Black-Litterman model: incorporate investor views.
        
        Views should be (n,) array of expected returns.
        Confidence should be variance of each view.
        """
        # Use market-cap-weighted returns as prior (simple version)
        if market_caps is None:
            market_caps = np.ones(self.n) / self.n
        
        # Prior expected returns (from CAPM or market data)
        mu_prior = self.mu_sample
        
        # View matrix P and view confidence Omega
        P = np.eye(self.n)  # Each row is a view (here: views on all assets)
        Omega = np.diag(view_confidence)
        
        # Black-Litterman posterior
        Sigma_inv = np.linalg.inv(self.sigma_sample + 1e-6 * np.eye(self.n))
        
        M = np.linalg.inv(Sigma_inv + P.T @ np.linalg.inv(Omega) @ P + 1e-8*np.eye(self.n))
        
        mu_bl = M @ (Sigma_inv @ mu_prior + P.T @ np.linalg.inv(Omega) @ views)
        
        return mu_bl
    
    def optimize_portfolio(self, mu: np.ndarray = None, sigma: np.ndarray = None,
                          target_return: float = None, long_only: bool = True) -> dict:
        """
        Optimize portfolio weights.
        """
        if mu is None:
            mu = self.mu_sample
        if sigma is None:
            sigma = self.sigma_sample
        
        if target_return is None:
            target_return = np.max(mu)
        
        def portfolio_variance(w):
            return w @ sigma @ w
        
        def portfolio_return(w):
            return w @ mu
        
        # Constraints
        constraints = [
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},  # Budget
            {'type': 'eq', 'fun': lambda w: portfolio_return(w) - target_return}  # Return
        ]
        
        # Bounds
        if long_only:
            bounds = [(0, 1) for _ in range(self.n)]
        else:
            bounds = [(None, None) for _ in range(self.n)]
        
        # Initial guess: equal weight
        w0 = np.ones(self.n) / self.n
        
        result = minimize(portfolio_variance, w0, method='SLSQP',
                         constraints=constraints, bounds=bounds)
        
        return {
            'weights': result.x,
            'variance': result.fun,
            'return': result.x @ mu,
            'sharpe': (result.x @ mu - self.rf) / np.sqrt(result.fun + 1e-8),
            'success': result.success
        }

# Generate synthetic market data
np.random.seed(42)
T = 252  # 1 year daily returns
n = 20   # 20 assets

# True parameters
true_mu = np.random.uniform(0.05, 0.15, n)
true_sigma_factor = np.random.randn(n, 5)
true_sigma = true_sigma_factor @ true_sigma_factor.T + np.eye(n) * 0.1
true_sigma = true_sigma / np.linalg.norm(true_sigma) * 2  # Scale

# Generate returns
returns = np.random.multivariate_normal(true_mu / 252, true_sigma / 252, T)

print(f"Generated market data: {T} observations, {n} assets")
print(f"Sample mean return: {returns.mean():.4f}")
print(f"Sample volatility: {returns.std():.4f}")

## Part 2: Shrinkage Estimation

### 2.1 Ledoit-Wolf Shrinkage Implementation & Comparison

In [ ]:
optimizer = PortfolioOptimizer(returns)

# Get different covariance estimators
sigma_shrunk, alpha = optimizer.ledoit_wolf_shrinkage()

print("\nCOVARIANCE ESTIMATION COMPARISON")
print("="*70)
print(f"Sample covariance condition number: {np.linalg.cond(optimizer.sigma_sample):.2e}")
print(f"Shrunk covariance condition number:  {np.linalg.cond(sigma_shrunk):.2e}")
print(f"Shrinkage intensity (α):            {alpha:.4f}")
print(f"\nCondition number improved by: {np.linalg.cond(optimizer.sigma_sample) / np.linalg.cond(sigma_shrunk):.1f}x")

## Part 3: Efficient Frontier

### 3.1 Compute Efficient Frontier with Different Estimators

In [ ]:
# Compute efficient frontier for different approaches
target_returns = np.linspace(optimizer.mu_sample.min(), optimizer.mu_sample.max(), 20)

results_sample = []  # Standard Markowitz
results_shrunk = []  # Ledoit-Wolf
results_bl = []      # Black-Litterman

for ret in target_returns:
    # Standard
    res = optimizer.optimize_portfolio(mu=optimizer.mu_sample, 
                                       sigma=optimizer.sigma_sample,
                                       target_return=ret)
    results_sample.append(res)
    
    # Shrinkage
    res = optimizer.optimize_portfolio(mu=optimizer.mu_sample,
                                       sigma=sigma_shrunk,
                                       target_return=ret)
    results_shrunk.append(res)
    
    # Black-Litterman (with weak views)
    views = optimizer.mu_sample + np.random.randn(n) * 0.01
    view_confidence = np.ones(n) * 0.01  # High confidence
    mu_bl = optimizer.black_litterman(views, view_confidence)
    res = optimizer.optimize_portfolio(mu=mu_bl,
                                       sigma=optimizer.sigma_sample,
                                       target_return=ret)
    results_bl.append(res)

# Extract data
vol_sample = [np.sqrt(r['variance']) for r in results_sample]
ret_sample = [r['return'] for r in results_sample]

vol_shrunk = [np.sqrt(r['variance']) for r in results_shrunk]
ret_shrunk = [r['return'] for r in results_shrunk]

vol_bl = [np.sqrt(r['variance']) for r in results_bl]
ret_bl = [r['return'] for r in results_bl]

print("Efficient frontiers computed.")

### 3.2 Visualization: Efficient Frontiers

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Efficient Frontiers Comparison
ax = axes[0, 0]
ax.plot(np.array(vol_sample)*100, np.array(ret_sample)*100, 'o-', linewidth=2, markersize=6,
        label='Standard (Sample Σ)', color='red')
ax.plot(np.array(vol_shrunk)*100, np.array(ret_shrunk)*100, 's-', linewidth=2, markersize=6,
        label='Ledoit-Wolf (Shrunk Σ)', color='blue')
ax.plot(np.array(vol_bl)*100, np.array(ret_bl)*100, '^-', linewidth=2, markersize=6,
        label='Black-Litterman', color='green')
ax.set_xlabel('Portfolio Volatility (%)', fontsize=11)
ax.set_ylabel('Expected Return (%)', fontsize=11)
ax.set_title('Efficient Frontiers: Multiple Estimation Methods', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Covariance Matrix Heatmap (Sample vs Shrunk)
ax = axes[0, 1]
im = ax.imshow(np.corrcoef(returns.T), cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
ax.set_title('Correlation Matrix (Estimated)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax)

# Plot 3: Portfolio Weights at Min-Variance
ax = axes[1, 0]
min_vol_idx = np.argmin(vol_sample)
weights_sample_mv = results_sample[min_vol_idx]['weights']
weights_shrunk_mv = results_shrunk[min_vol_idx]['weights']

x = np.arange(10)  # Show first 10 assets
width = 0.35
ax.bar(x - width/2, weights_sample_mv[:10]*100, width, label='Sample', alpha=0.8, color='red')
ax.bar(x + width/2, weights_shrunk_mv[:10]*100, width, label='Shrinkage', alpha=0.8, color='blue')
ax.set_xlabel('Asset', fontsize=11)
ax.set_ylabel('Weight (%)', fontsize=11)
ax.set_title('Min-Variance Portfolio: Weights (First 10 Assets)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: Sharpe Ratio Comparison
ax = axes[1, 1]
sharpe_sample = [(r['return'] - 0.02) / np.sqrt(r['variance']) for r in results_sample]
sharpe_shrunk = [(r['return'] - 0.02) / np.sqrt(r['variance']) for r in results_shrunk]
sharpe_bl = [(r['return'] - 0.02) / np.sqrt(r['variance']) for r in results_bl]

ax.plot(target_returns*100, sharpe_sample, 'o-', linewidth=2, label='Sample', color='red')
ax.plot(target_returns*100, sharpe_shrunk, 's-', linewidth=2, label='Shrinkage', color='blue')
ax.plot(target_returns*100, sharpe_bl, '^-', linewidth=2, label='Black-Litterman', color='green')
ax.set_xlabel('Target Portfolio Return (%)', fontsize=11)
ax.set_ylabel('Sharpe Ratio', fontsize=11)
ax.set_title('Risk-Adjusted Performance (Sharpe Ratio)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('portfolio_optimization_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSHARPE RATIO COMPARISON")
print("="*70)
print(f"Best Sharpe (Sample):      {max(sharpe_sample):.4f}")
print(f"Best Sharpe (Shrinkage):   {max(sharpe_shrunk):.4f}")
print(f"Best Sharpe (Black-Litterman): {max(sharpe_bl):.4f}")
print(f"\nImprovement (Shrinkage):   {(max(sharpe_shrunk) - max(sharpe_sample))/max(sharpe_sample)*100:.1f}%")

## Part 4: Robustness to Estimation Error

### 4.1 Simulation: How do different methods perform out-of-sample?


In [ ]:
# Out-of-sample test
np.random.seed(43)  # Different seed
test_returns = np.random.multivariate_normal(true_mu / 252, true_sigma / 252, 126)  # 6 months

# Get min-variance portfolios from training data
res_sample_test = optimizer.optimize_portfolio(mu=optimizer.mu_sample,
                                              sigma=optimizer.sigma_sample,
                                              target_return=np.mean(optimizer.mu_sample))
res_shrunk_test = optimizer.optimize_portfolio(mu=optimizer.mu_sample,
                                              sigma=sigma_shrunk,
                                              target_return=np.mean(optimizer.mu_sample))
res_equal_weight = {'weights': np.ones(n) / n}

# Evaluate on test data
def evaluate_portfolio(weights, test_ret):
    port_ret = test_ret @ weights
    mean_ret = np.mean(port_ret)
    vol = np.std(port_ret)
    sharpe = (mean_ret - 0.02/252) / vol if vol > 0 else 0
    return mean_ret, vol, sharpe

mean_sample, vol_sample_test, sharpe_sample_test = evaluate_portfolio(res_sample_test['weights'], test_returns)
mean_shrunk, vol_shrunk_test, sharpe_shrunk_test = evaluate_portfolio(res_shrunk_test['weights'], test_returns)
mean_equal, vol_equal, sharpe_equal = evaluate_portfolio(res_equal_weight['weights'], test_returns)

print("\nOUT-OF-SAMPLE PERFORMANCE (Test Set)")
print("="*70)
print(f"{'Method':<20} {'Return':>15} {'Volatility':>15} {'Sharpe':>15}")
print("-"*70)
print(f"{'Equal Weight':<20} {mean_equal*252:>14.2%} {vol_equal*np.sqrt(252):>14.2%} {sharpe_equal*np.sqrt(252):>14.4f}")
print(f"{'Sample Markowitz':<20} {mean_sample*252:>14.2%} {vol_sample_test*np.sqrt(252):>14.2%} {sharpe_sample_test*np.sqrt(252):>14.4f}")
print(f"{'Shrinkage':<20} {mean_shrunk*252:>14.2%} {vol_shrunk_test*np.sqrt(252):>14.2%} {sharpe_shrunk_test*np.sqrt(252):>14.4f}")

## Conclusions

1. **Estimation error dominates**: Sample covariance is severely ill-conditioned ($\kappa \gg 1$) in high dimensions.

2. **Shrinkage improves robustness**: Ledoit-Wolf dramatically reduces condition number and improves out-of-sample performance.

3. **Black-Litterman adds realism**: Incorporates investor views as a form of regularization, preventing extreme weights.

4. **Theory meets practice**: Markowitz won the Nobel Prize, but practitioners found it unusable until Ledoit-Wolf shrinkage.

### Key Takeaway
> **"The elegant mathematical formulation hides the practical challenge: estimating parameters accurately. Modern portfolio optimization is 80% about estimation, 20% about optimization."**
